In [ ]:
import os

# CELL 0 - GPU CHECK
os.system("nvidia-smi")
!nvidia-smi


In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
import os
print(os.listdir())

In [ ]:
os.chdir("VANSHIKASOHAL")
print(os.listdir())

In [ ]:
import zipfile
with zipfile.ZipFile("DiffMOT-main.zip",'r') as zip_ref:
    zip_ref.extractall()
print("extracted")

In [ ]:
# CELL 1 - CLONE DiffMOT REPO  ← CHANGED (was DiffusionTrack)
#import os
#if not os.path.exists("DiffMot"):
 #  os.system("git clone https://github.com/Kroery/DiffMOT.git")
#os.chdir("DiffMOT")#print("DiffMOT repo cloned!")


In [ ]:
import os

# Install PyTorch (auto compatible)
os.system("pip install torch torchvision torchaudio")

# Main requirements
os.system("pip install -r requirement.txt")

# YOLOX (needed internally)
os.chdir("external/YOLOX")
os.system("pip install -r requirements.txt")
os.system("pip install -e .")

# deep-person-reid
os.chdir("../deep-person-reid")
os.system("pip install -r requirements.txt")
os.system("pip install -e .")

# fast_reid
os.chdir("../fast_reid")
os.system("pip install -r docs/requirements.txt")

# back to main
os.chdir("../../")

# YOLOv8
os.system("pip install ultralytics")

print("ALL INSTALLED")

In [ ]:
# CELL 3 - TORCH CHECK
import torch
print(torch.__version__)
print(torch.cuda.is_available())


In [ ]:
import os
import zipfile
import shutil
import urllib.request

# STEP 1: Go to DiffMOT folder
#os.chdir("DiffMOT-main")

# STEP 2: Create dataset folders
os.makedirs("datasets/mot/train", exist_ok=True)
os.makedirs("datasets/mot/test", exist_ok=True)

# STEP 3: Download MOT17
print("Downloading MOT17 dataset (~5GB, will take time)...")

url = "https://motchallenge.net/data/MOT17.zip"
zip_path = "datasets/mot/MOT17.zip"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
    print("Download complete!")
else:
    print("Dataset already downloaded.")

# STEP 4: Extract dataset
print("Extracting dataset...")

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("datasets/mot/")

# STEP 5: Move files to correct structure
print("Organizing dataset...")

train_src = "datasets/mot/MOT17/train"
test_src  = "datasets/mot/MOT17/test"

train_dst = "datasets/mot/train"
test_dst  = "datasets/mot/test"

for item in os.listdir(train_src):
    src = os.path.join(train_src, item)
    dst = os.path.join(train_dst, item)
    if not os.path.exists(dst):
        shutil.move(src, dst)

for item in os.listdir(test_src):
    src = os.path.join(test_src, item)
    dst = os.path.join(test_dst, item)
    if not os.path.exists(dst):
        shutil.move(src, dst)

# STEP 6: Cleanup
shutil.rmtree("datasets/mot/MOT17")
os.remove(zip_path)

print(" MOT17 dataset ready!")

# STEP 7: Verify
print("\nTrain sequences:")
print(os.listdir("datasets/mot/train"))


In [ ]:
import os
print(os.path.exists("datasets/mot/train"))
import os
print(os.listdir("datasets/mot/train"))

In [ ]:
import os
#os.chdir("DiffMOT-main")
ret = os.system("python mot_data_process.py")
if ret != 0:
    print("WARNING: mot_data_process.py had errors - possibly missing MOT20, may be okay if MOT17 processed fine")
    # Manually verify
    import os
    print(os.listdir("datasets/mot/train"))
print("Data preprocessing done!")

In [ ]:
import os
import urllib.request

# go inside project (if not already)
#os.chdir("DiffMOT-main")

# create folder
os.makedirs("pretrained/motion", exist_ok=True)

# download file
url = "https://github.com/Kroery/DiffMOT/releases/download/v1.0/MOT_epoch800.pt"
output_path = "pretrained/motion/MOT_epoch800.pt"

if not os.path.exists(output_path):
    print("Downloading motion model...")
    urllib.request.urlretrieve(url, output_path)
    print("Download complete!")
else:
    print("Model already exists.")

# check file size
size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"File size: {size_mb:.2f} MB")

print("Motion model ready!")


In [ ]:
import os
import urllib.request

# go inside project (if not already)
#os.chdir("DiffMOT-main")

# create folder
os.makedirs("pretrained/reid", exist_ok=True)

# download file
url = "https://github.com/Kroery/DiffMOT/releases/download/v1.0/mot17_sbs_S50.pth"
output_path = "pretrained/reid/mot17_sbs_S50.pth"

if not os.path.exists(output_path):
    print("Downloading ReID model...")
    urllib.request.urlretrieve(url, output_path)
    print("Download complete!")
else:
    print("Model already exists.")

# check file size
size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"File size: {size_mb:.2f} MB")

print("ReID model ready!")

In [ ]:
import os
import numpy as np
import torch
from ultralytics import YOLO

# go to project folder
#os.chdir("DiffMOT-main")

# CONFIG
MODEL_WEIGHTS = "yolov8n.pt"#check
CONF_THRESH   = 0.3
IOU_THRESH    = 0.45
IMG_SIZE      = 1280
PERSON_CLASS  = 0

# device selection
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = YOLO(MODEL_WEIGHTS)
model.to(device)

# MOT17 sequences
seqs = ['MOT17-02-FRCNN','MOT17-04-FRCNN','MOT17-05-FRCNN','MOT17-09-FRCNN','MOT17-10-FRCNN','MOT17-11-FRCNN','MOT17-13-FRCNN']

for seq in seqs:
    img_dir  = f"datasets/mot/train/{seq}/img1"
    det_dir  = f"datasets/mot/train/{seq}/det"
    det_path = f"{det_dir}/det.txt"

    if not os.path.exists(img_dir):
        print(f"Skipping {seq}, folder not found")
        continue

    os.makedirs(det_dir, exist_ok=True)

    frames = sorted(os.listdir(img_dir))
    print(f"\n[{seq}] Running YOLOv8 on {len(frames)} frames...")

    lines = []

    for frame_file in frames:
        frame_id = int(os.path.splitext(frame_file)[0])
        img_path = os.path.join(img_dir, frame_file)

        results = model.predict(
            source  = img_path,
            conf    = CONF_THRESH,
            iou     = IOU_THRESH,
            imgsz   = IMG_SIZE,
            classes = [PERSON_CLASS],
            verbose = False,
            device  = device
        )

        boxes = results[0].boxes

        if boxes is not None and len(boxes) > 0:
            xyxy  = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()

            for (x1, y1, x2, y2), conf in zip(xyxy, confs):
                w = x2 - x1
                h = y2 - y1
                lines.append(f"{frame_id},-1,{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1")

    with open(det_path, "w") as f:
        f.write("\n".join(lines))

    print(f"Saved {len(lines)} detections to {det_path}")

print("YOLOv8 detections complete")


In [ ]:
import os
for seq in seqs:
    path=f"datasets/mot/train/{seq}/det/det.txt"
    print(seq,"->",os.path.exists(path))

In [ ]:
import os

#os.chdir("DiffMOT-main")

seqs = ['MOT17-02-FRCNN','MOT17-04-FRCNN','MOT17-05-FRCNN','MOT17-09-FRCNN','MOT17-10-FRCNN','MOT17-11-FRCNN','MOT17-13-FRCNN']

print("Detection file check:")

for seq in seqs:
    det_path = f"datasets/mot/train/{seq}/det/det.txt"

    if os.path.exists(det_path):
        lines = open(det_path).readlines()
        print(f"{seq}: {len(lines)} detections")

        if len(lines) > 0:
            print("Sample:", lines[0].strip())
        else:
            print("EMPTY FILE")

    else:
        print(f"{seq}: det.txt NOT FOUND")


In [ ]:
import yaml
import os

#os.chdir("DiffMOT-main")

config_path = "configs/mot17_test.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Original config:")
print(yaml.dump(config))


# FIXED PATHS (LOCAL)

config["det_dir"]    = "datasets/mot/train"
config["reid_dir"]   = "pretrained/reid"
config["model_path"] = "pretrained/motion/MOT_epoch800.pt"
config["info_dir"]   = "datasets/mot/train"
config["save_dir"]   = "outputs/mot17"

config["device"]="cuda"
config["gpus"]=[0]
# thresholds (keep same)
config["high_thres"]  = 0.6
config["low_thres"]   = 0.1
config["w_assoc_emb"] = 2.2
config["aw_param"]    = 1.7
config["batch_size"]=512

# save config
with open(config_path, "w") as f:
    yaml.dump(config, f)

print("Updated config saved\n")

with open(config_path) as f:
    print(f.read())


In [ ]:
import os
print(os.listdir("datasets/mot/train"))

In [ ]:
print(open("configs/mot17_test.yaml").read())

In [ ]:
import torch
print("CUDA AVAILABLE",torch.cuda.is_available())
print("GPU COUNT:",torch.cuda.device_count())
print("GPU NAME:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
!pip install tensorboardX

In [ ]:
!pip install lap cython bbox motmetrics scipy opencv-python tqdm matplotlib scikit-learn filterpy

In [ ]:
!pip install cython_bbox==0.1.3

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
import os

for root, dirs, files in os.walk("."):
    print("\n📁", root)
    for d in dirs:
        print("  📂", d)
    for f in files:
        print("  📄", f)

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
print(os.listdir("DiffMOT-main"))

In [ ]:
import os

os.makedirs("cython_bbox", exist_ok=True)

with open("cython_bbox/__init__.py", "w") as f:
    f.write("""
import numpy as np

def bbox_overlaps(boxes1, boxes2):
    overlaps = np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)
    for i, box1 in enumerate(boxes1):
        x1, y1, x2, y2 = box1
        area1 = (x2 - x1) * (y2 - y1)

        for j, box2 in enumerate(boxes2):
            xx1 = max(x1, box2[0])
            yy1 = max(y1, box2[1])
            xx2 = min(x2, box2[2])
            yy2 = min(y2, box2[3])

            w = max(0, xx2 - xx1)
            h = max(0, yy2 - yy1)
            inter = w * h

            area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
            union = area1 + area2 - inter

            overlaps[i, j] = inter / union if union > 0 else 0

    return overlaps
""")

print("Dummy cython_bbox module created!")


In [ ]:
!pip install torchreid

In [ ]:
!pip install easydict

In [ ]:
import os 
import yaml
config["eval_expname"]=" "

In [ ]:
import os
path="petrained/motion/MOT_epoch800.pt"
print("Exist:",os.path.exists(path))
print("Full path",os.path.abspath(path))

In [ ]:
import urllib.request
import os

os.makedirs("pretrained/motion", exist_ok=True)

url = "https://github.com/Kroery/DiffMOT/releases/download/v1.0/MOT_epoch800.pt"
path = "pretrained/motion/MOT_epoch800.pt"

urllib.request.urlretrieve(url, path)

print("Downloaded successfully!")


In [ ]:
import os
path="petrained/motion/MOT_epoch800.pt"
print("Exist:",os.path.exists(path))
print("Full path",os.path.abspath(path))

In [ ]:
import urllib.request
import os

os.makedirs("pretrained/motion", exist_ok=True)

url = "https://github.com/Kroery/DiffMOT/releases/download/v1.0/MOT_epoch800.pt"
path = "pretrained/motion/MOT_epoch800.pt"

urllib.request.urlretrieve(url, path)

print("Downloaded successfully!")
import os
path="pretrained/motion/MOT_epoch800.pt"
print("Exist:", os.path.exists(path))
print("Full path:", os.path.abspath(path))
import urllib.request
import os

os.makedirs("pretrained/motion", exist_ok=True)

url = "https://github.com/Kroery/DiffMOT/releases/download/v1.0/MOT_epoch800.pt"
path = "pretrained/motion/MOT_epoch800.pt"

try:
    urllib.request.urlretrieve(url, path)
    if os.path.exists(path):
        print("Download verified")
    else:
        print("Download failed")
except Exception as e:
    print("Error:", e)
import requests

url = "https://github.com/Kroery/DiffMOT/releases/download/v1.0/MOT_epoch800.pt"
path = "pretrained/motion/MOT_epoch800.pt"

response = requests.get(url, stream=True)
with open(path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print("Done. Exists:", os.path.exists(path))


In [ ]:
import os

path = "pretrained/motion/MOT_epoch800.pt"
print("File size (MB):", os.path.getsize(path) / (1024*1024))


In [ ]:
import os
print(os.listdir("models"))

In [ ]:
import os
print(os.getcwd())

In [ ]:
os.chdir("DiffMOT-main")
print(os.getcwd())

In [ ]:
!pip install torch torchvision torchaudio
!pip install ultralytics
!pip install tensorboardX lap motmetrics scipy opencv-python tqdm matplotlib scikit-learn filterpy torchreid gdown

In [ ]:
import os

paths = [
    "pretrained/motion/MOT_epoch800.pt",
    "pretrained/reid/mot17_sbs_S50.pth",
    "datasets/mot/train"
]

for p in paths:
    print(p, "→", os.path.exists(p))

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
import os

print("pretrained exists:", os.path.exists("pretrained"))
print("datasets exists:", os.path.exists("datasets"))

In [ ]:
import os
os.listdir()

In [ ]:
!python  main.py

In [ ]:
pip install -r requirement.txt

In [ ]:
import os
print(os.listdir("dataset"))

In [ ]:
import os

for root, dirs, files in os.walk("."):
    if "det.txt" in files:
        print(root)

In [ ]:
#new
import os

model_path = "pretrained/motion/MOT_epoch800.pt"

print("Model exists:", os.path.exists(model_path))

# list pretrained folder
print("\nFiles in pretrained/motion:")
print(os.listdir("pretrained/motion"))

In [ ]:
import os

os.makedirs("pretrained/motion", exist_ok=True)
os.makedirs("pretrained/reid", exist_ok=True)

# Motion model
os.system('curl -L -o pretrained/motion/MOT_epoch800.pt https://github.com/Kroery/DiffMOT/releases/download/v1.0/MOT_epoch800.pt')

# ReID model
os.system('curl -L -o pretrained/reid/mot17_sbs_S50.pth https://github.com/Kroery/DiffMOT/releases/download/v1.0/mot17_sbs_S50.pth')

In [ ]:
import torch

path = "pretrained/motion/MOT_epoch800.pt"

model = torch.load(path, map_location="cpu")
print("Model loaded successfully!")
print(type(model))

In [ ]:
!pip install einops

In [ ]:
!pip install cython_bbox

In [ ]:
import sys
print(sys.executable)

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
!pip install gdown


In [ ]:
!pip install tensorboard

In [ ]:
!pip install numpy scipy matplotlib opencv-python tqdm pillow scikit-learn lap filterpy motmetrics
!pip install lap
!pip install filterpy
!pip install motmetrics

In [ ]:
!pip install yacs

In [ ]:
!pip install termcolor

In [ ]:
!pip install einops pyyaml argparse easydict numpy tensorboardX tqdm opencv-python scipy lap  fvcore

In [ ]:
from external.adaptors.fastreid_adaptor import FastReID
print("IMPORT OK ")

In [ ]:
!python main.py --config configs/mot.yaml --dataset MOT17

In [ ]:
!python main.py \
--dataset mot \
--data_dir datasets \
--dataset_name MOT17 \
--mode test \
--device cuda \
--batch_size 1 \
--exp_name mot_ddm_1000_deeper \
--detector FRCNN

In [ ]:
import os
print(os.getcwd())
os.chdir("DiffMOT-main")
print(os.getcwd())

In [ ]:
!pip install fastreid


In [ ]:
import sys
import os

# project root
project_path = r"C:\Users\vansh\VANSHIKASOHAL\DiffMOT-main"

# add all required paths
sys.path.insert(0, project_path)
sys.path.insert(0, os.path.join(project_path, "external"))
sys.path.insert(0, os.path.join(project_path, "external", "fast_reid"))
sys.path.insert(0, os.path.join(project_path, "external", "fast_reid", "fastreid"))

print("PATH SET ")

In [ ]:
import os
print(os.getcwd())
os.chdir("DiffMOT-main")
print(os.getcwd())

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

In [1]:
print("HELLO")

HELLO


In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
model.to(device)

In [ ]:
from external.adaptors.fastreid_adaptor import FastReID
print("IMPORT SUCCESS ")

In [ ]:
from fastreid.config import get_cfg
from external.adaptors.fastreid_adaptor import FastReID

print("ALL IMPORTS WORKING ")

In [ ]:
from diffmot import DiffMOT

import yaml
with open("configs/mot.yaml", 'r') as f:
    config = yaml.safe_load(f)

agent = DiffMOT(config)
agent.eval()

In [ ]:
import os
!pip install trackeval -q
#os.chdir("DiffMOT-main")

os.system("""python -m trackeval.scripts.run_mot_challenge \
    --SPLIT_TO_EVAL train \
    --METRICS HOTA CLEAR Identity \
    --GT_FOLDER datasets/mot/train \
    --TRACKERS_FOLDER outputs/mot17 \
    --TRACKER_SUB_FOLDER . \
    --USE_PARALLEL False \
    --NUM_PARALLEL_CORES 1 \
    --PRINT_RESULTS True \
    --OUTPUT_SUMMARY True \
    --OUTPUT_EMPTY_CLASSES False \
    --SKIP_SPLIT_FOL True""")

In [ ]:
import glob

print("YOLOv8 Detection + DiffMOT Tracking — MOT17 RESULTS")

summary_files = glob.glob("outputs/mot17/**/*summary*.txt", recursive=True)
if summary_files:
    for sf in summary_files:
        print(f"\n--- {sf} ---")
        print(open(sf).read())
else:
    print("No summary files yet. All output files:")
    for f in glob.glob("outputs/mot17/**/*.txt", recursive=True):
        print(f"  {f}")

In [ ]:
import os
print(os.path.exists("pretrained/motion/MOT_epoch800.pt"))

In [ ]:
import os
os.system("python main.py --config configs/mot17_test.yaml")

In [ ]:
import os

# current working directory
print("Current Path:", os.getcwd())

# check important folders
print("\nFolders in root:")
print(os.listdir())

# check dataset path
dataset_path = "datasets/mot/train"
print("\nDataset exists:", os.path.exists(dataset_path))

# list few dataset folders
if os.path.exists(dataset_path):
    print("\nSample dataset folders:")
    print(os.listdir(dataset_path)[:5])

In [ ]:
import os

os.chdir("DiffMOT-main")

print("Now Current Path:", os.getcwd())
print(os.listdir())

In [ ]:
import os

base_path = "datasets/mot/train"

folders = [f for f in os.listdir(base_path) if "FRCNN" in f]

print("FRCNN Folders:\n")

for f in folders:
    path = os.path.join(base_path, f)
    
    has_img = os.path.exists(os.path.join(path, "img1"))
    has_det = os.path.exists(os.path.join(path, "det"))
    
    print(f"{f} --> img1: {has_img}, det: {has_det}")